<div style="border-radius: 10px; padding: 32px 0px; border: 1px solid rgba(128,128,128,0.2);">

  <div style="display: flex; justify-content: space-between; align-items: flex-start; flex-wrap: wrap; gap: 16px; padding: 0px 32px;">

  <div>
    <div style="font-size: 0.75rem; letter-spacing: 3px; text-transform: uppercase; font-weight: 600; margin-bottom: 10px; opacity: 0.6;">
      Máster Universitario en Big Data y Computación en la Nube.
    </div>
    <div style="font-size: 1.5rem; font-weight: 700; margin-bottom: 4px;">
      Trabajo de Fin de Máster
    </div>
    <div style="font-size: 1rem; font-weight: 400; opacity: 0.75;">
      Clasificador taxonómico de boletines oficiales españoles
    </div>
  </div>

  <div style="margin-top: 16px; display: flex; align-items: center; gap: 12px;">
    <div style="font-size: 1rem; font-weight: 600;">Hugo de Lamo</div>
  </div>

  </div>
</div>

# 05 · Clasificador taxonómico de boletines oficiales

Este notebook implementa un clasificador multietiqueta de publicaciones de boletines oficiales españoles usando **Pydantic AI**. El problema es una aguja en un pajar: de ~65 000 publicaciones del Q1 2025, solo ~3,7 % son relevantes para el dominio ambiental-energético.

El clasificador responde cuatro preguntas por publicación:
1. **¿Es relevante?** — ¿Pertenece al universo de autorizaciones ambiental-energéticas?
2. **¿Qué procedimientos contiene?** — Lista multilabel: DIA, AAP, AAC, AAU, IIA, AAI, IAE, AEX.
3. **¿En qué fase está?** — ¿Resolución final o trámite en curso?
4. **¿Qué tecnología menciona?** — Lista multilabel: fotovoltaica, eólica, biogás_biomasa…

---

## Estructura del notebook

| § | Sección | Contenido |
|---|---------|----------|
| **0** | **Setup** | Entorno, dependencias, modelo |
| **1** | **Schema de output** | `ClassifierOutput`, enums y reglas de negocio |
| **2** | **Ground truth** | Construcción del dataset etiquetado manualmente |
| **3** | **Agente clasificador** | System prompt, agent con Pydantic AI, run single |
| **4** | **Baseline** | Experimento 0 — claude-sonnet-4-6 sobre ground truth |
| **5** | **Experimentos de ablación** | Efecto de `phase`, few-shot y campo `technologies` |
| **6** | **Modelos locales** | Experimento 4 — modelos 7B via LM Studio |
| **7** | **Análisis y conclusiones** | Comparativa de métricas, falsos positivos, siguientes pasos |

---

## §0. Setup

Cargamos las variables de entorno (la API key de Gemini vive en `.env`, nunca en el código) e importamos las librerías del proyecto.

In [ ]:
import os

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent

load_dotenv()

In [ ]:
# Único punto de cambio para conectar otro proveedor.
# Para LM Studio (modelos locales): "openai:nombre-del-modelo" con base_url="http://127.0.0.1:1234/v1"
MODEL = "gemini-2.5-pro"

In [ ]:
api_key = os.getenv("GEMINI_API_KEY")
assert api_key, "GEMINI_API_KEY no encontrada — revisa el archivo .env"

print("Entorno listo.")
print(f"  Modelo:  {MODEL}")
print(f"  API key: {api_key[:8]}{'*' * (len(api_key) - 8)}")